<div
  style="
    background-color: #f0f0f0;
    color:rgb(56, 56, 56);
    padding: 8px;
    display: flex;
    align-items: center;
    gap: 100px;
  "
>
  <img src="../../../images/brand.svg" style="max-height: 80px;">
  <strong>
    AI Saga: Data Science and Machine Learning</br>
    6.lab. Unsupervised Learning: Wisconsin Cancer Dataset
  </strong>
</div>

### Background

You will work with the Wisconsin Diagnostic Cancer Dataset to explore unsupervised learning techniques using scikit-learn.

The dataset includes:

- 569 instances
- 30 numeric features computed from the cell nuclei present in the image
- We will **ignore the class labels** for this unsupervised learning exercise

### Description

Your task is to:

- Apply PCA to reduce the dimensionality of the data
- Implement K-means clustering to group similar samples
- Compare the clusters with the real diagnoses
- Answer clearly:
  - What is the minimum number of principal components needed to explain 95% of the variance?
  - What is the variance explained by the first two principal components?
  - Visually inspect the groups: could they be easily separated by a linear boundary? (malignant vs benign)

### Deliverables

- An **ipynb** (Jupyter Notebook) file called **wisconsin-cancer-unsupervised.ipynb**
- The notebook should include:
  - PCA implementation and analysis
  - K-means clustering implementation
  - Required visualizations
  - A discussion on how the unsupervised methods captured the underlying structure
- Use the provided template as a starting point

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import (homogeneity_score, completeness_score,
                             v_measure_score, adjusted_rand_score)

os.makedirs('data', exist_ok=True)
np.random.seed(42)

# 1. Carregar e Explorar o Dataset

Carregamos o dataset usando as features. Os rótulos (`target`) são guardados
separadamente e **não** são usados em nenhum momento do PCA ou do K-means: eles
servem somente para comparar os agrupamentos no final.

In [ ]:
raw = load_breast_cancer(as_frame=True)
X = raw.data
y_true = raw.target            # guardado apenas para comparação posterior
target_names = raw.target_names

print(f'Dataset shape: {X.shape}')
print(f'Instances: {X.shape[0]} | Features: {X.shape[1]}')
print(f'\nFeature names ({len(raw.feature_names)}):')
print(list(raw.feature_names))
print(f'\nClasse real (apenas referência): {dict(zip(target_names, y_true.value_counts().sort_index()))}')

X.head()

# 2. Padronizar os Dados

As 30 features têm escalas muito diferentes (por exemplo `mean area` na casa dos
milhares e `mean smoothness` na casa dos milésimos). Como PCA e K-means são
baseados em variância/distância, padronizamos cada feature para média 0 e desvio
padrão 1 antes de aplicá-los.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Média por feature (após padronização):')
print(np.round(X_scaled.mean(axis=0), 6))
print('\nDesvio padrão por feature (após padronização):')
print(np.round(X_scaled.std(axis=0), 6))

# 3. Análise de Componentes Principais (PCA)

Ajustamos o PCA com todas as 30 componentes para poder ver a variância explicada
por cada uma e a variância acumulada.

In [ ]:
pca = PCA(n_components=X_scaled.shape[1])
pca.fit(X_scaled)

explained = pca.explained_variance_ratio_
cumulative = np.cumsum(explained)

pca_table = pd.DataFrame({
    'component': np.arange(1, len(explained) + 1),
    'explained_variance_ratio': explained,
    'cumulative_variance': cumulative,
})

print(pca_table.round(4).to_string(index=False))
pca_table.round(6).to_csv('data/pca_explained_variance.csv', index=False)

### Respostas do PCA

**1. Número mínimo de componentes para explicar 95% da variância**

**2. Variância explicada pelos dois primeiros componentes principais**

In [ ]:
n_components_95 = int(np.argmax(cumulative >= 0.95) + 1)
variance_two = cumulative[1]

print(f'1) Componentes mínimos para explicar 95% da variância: {n_components_95}')
print(f'   (variância acumulada com {n_components_95} componentes: {cumulative[n_components_95 - 1]:.4%})')
print(f'2) Variância explicada pelos dois primeiros componentes: {variance_two:.4%}')
print(f'   PC1 isolado: {explained[0]:.4%} | PC2 isolado: {explained[1]:.4%}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(np.arange(1, len(cumulative) + 1), cumulative, marker='o', label='Variância acumulada')
ax.axhline(0.95, color='red', linestyle='--', label='Limiar de 95%')
ax.axvline(n_components_95, color='green', linestyle='--',
           label=f'{n_components_95} componentes')
ax.set_xlabel('Número de componentes principais')
ax.set_ylabel('Variância explicada acumulada')
ax.set_title('Variância explicada acumulada por número de componentes')
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('data/pca_cumulative_variance.png', dpi=150, bbox_inches='tight')
plt.show()

# 4. Agrupamento K-Means

Como o método é não supervisionado, escolhemos `k = 2` porque o dataset descreve
dois grupos biológicos (maligno e benigno). Ajustamos o K-means sobre os dados
padronizados e comparamos os grupos resultantes com os diagnósticos reais.

In [ ]:
kmeans = KMeans(n_clusters=2, n_init=10, random_state=42)
cluster_labels = kmeans.fit_predict(X_scaled)

print('Tamanho dos clusters:')
print(pd.Series(cluster_labels).value_counts().sort_index().to_string())

contingency = pd.crosstab(
    pd.Series(cluster_labels, name='cluster'),
    pd.Series(y_true, name='diagnostico_real'),
)
contingency.columns = list(target_names)
contingency.to_csv('data/cluster_vs_diagnosis.csv')

print('\nTabela de contingência (cluster x diagnóstico real):')
print(contingency.to_string())

In [ ]:
metrics = pd.DataFrame({
    'metrica': ['Homogeneity', 'Completeness', 'V-measure', 'Adjusted Rand Index'],
    'valor': [
        homogeneity_score(y_true, cluster_labels),
        completeness_score(y_true, cluster_labels),
        v_measure_score(y_true, cluster_labels),
        adjusted_rand_score(y_true, cluster_labels),
    ],
})

print(metrics.round(4).to_string(index=False))
metrics.round(6).to_csv('data/cluster_metrics.csv', index=False)

# 5. Comparação Visual dos Grupos

Projetamos as amostras nos dois primeiros componentes principais e plotamos duas
vezes: colorido pelo diagnóstico real e colorido pelo cluster do K-means. Assim
dá para inspecionar visualmente se os grupos são separáveis por uma fronteira
linear.

In [ ]:
X_pca2 = pca.transform(X_scaled)[:, :2]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

scatter_a = axes[0].scatter(X_pca2[:, 0], X_pca2[:, 1], c=y_true,
                            cmap='coolwarm', edgecolors='k', s=30)
axes[0].set_title('Diagnóstico real')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
cbar_a = fig.colorbar(scatter_a, ax=axes[0], ticks=[0, 1])
cbar_a.ax.set_yticklabels(target_names)

scatter_b = axes[1].scatter(X_pca2[:, 0], X_pca2[:, 1], c=cluster_labels,
                            cmap='coolwarm', edgecolors='k', s=30)
axes[1].set_title('Grupos K-means (k = 2)')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
cbar_b = fig.colorbar(scatter_b, ax=axes[1], ticks=[0, 1])
cbar_b.ax.set_yticklabels(['Cluster 0', 'Cluster 1'])

plt.suptitle('Wisconsin Cancer: Projeção nos dois primeiros componentes principais', fontsize=14)
plt.tight_layout()
plt.savefig('data/pca_scatter_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))
scatter = ax.scatter(X_pca2[:, 0], X_pca2[:, 1], c=y_true,
                     cmap='coolwarm', edgecolors='k', s=30)
ax.axvline(0, color='gray', linestyle='--', alpha=0.7)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_title('Diagnóstico real com fronteira linear (PC1 = 0)')
cbar = fig.colorbar(scatter, ax=ax, ticks=[0, 1])
cbar.ax.set_yticklabels(target_names)
plt.tight_layout()
plt.savefig('data/pca_scatter_linear_boundary.png', dpi=150, bbox_inches='tight')
plt.show()

# 6. Discussão: Os Métodos Não Supervisionados Capturaram a Estrutura?

### 6.1 Quantas componentes para 95% da variância

O PCA mostrou que as 30 features originais são bastante redundantes. São
necessárias **10 componentes principais** para explicar 95% da variância. Ou
seja, dá para reduzir de 30 para 10 dimensões perdendo menos de 5% da
informação. As primeiras componentes explicam: PC1 = 44.3%, PC2 = 19.0%,
PC3 = 9.4%, caindo rápido depois disso. Isso é esperado nesse dataset, porque
várias medidas são versões de uma mesma grandeza (por exemplo `mean radius`,
`mean area` e `mean perimeter` são altamente correlacionadas).

### 6.2 Variância dos dois primeiros componentes

Os dois primeiros componentes explicam juntos **63.2% da variância** (44.3% no
PC1 e 19.0% no PC2). Por isso a projeção 2D usada nos gráficos já mostra a
estrutura principal dos dados, apesar de ainda restar cerca de 37% da variância
em dimensões não visualizadas.

### 6.3 Os grupos são separados por uma fronteira linear?

Sim, em grande parte. Olhando o gráfico colorido pelo diagnóstico real, as
amostras benignas e malignas se organizam quase em duas nuvens separadas ao
longo do PC1: os casos malignos tendem a PC1 positivo e os benignos a PC1
negativo. Uma única linha vertical (PC1 = 0) já acerta cerca de 92% das
amostras, o que mostra que uma fronteira linear resolve a maior parte do
problema. Existe, porém, uma zona de sobreposição: alguns tumores malignos caem
no lado benigno e vice-versa, e são justamente esses os casos ambíguos. Então a
resposta honesta é: **sim, poderiam ser majoritariamente separados por uma
fronteira linear, mas não perfeitamente**, e há uma faixa de mistura entre as
classes.

### 6.4 O que o K-means encontrou

Sem nunca ter visto os rótulos, o K-means (k = 2) encontrou dois grupos de
tamanhos 375 e 194. Comparando com os diagnósticos reais:

- Cluster 0: 339 benignos e 36 malignos (dominantemente benigno)
- Cluster 1: 176 malignos e 18 benignos (dominantemente maligno)

O Adjusted Rand Index foi de 0.65 e o V-measure de 0.53, indicando uma
concordância substancial com os diagnósticos reais, mas não perfeita. Em outras
palavras, o agrupamento natural dos dados *corresponde* à distinção
maligno/benigno, com os erros concentrados nos casos que ficam na fronteira
entre os dois grupos.

### 6.5 Como os métodos não supervisionados captaram a estrutura

O ponto mais interessante é que nem o PCA nem o K-means usaram os rótulos, e
mesmo assim os dois convergiram para a mesma estrutura latente:

- O **PCA** revelou que a maior parte da variação dos dados está concentrada em
  poucas direções. O PC1, que carrega 44% da variância, é a direção que mais
  separa os dois tipos de tumor (tamanho e irregularidade do núcleo celular).
  Ou seja, a variação "natural" dos dados coincide com a diferença entre maligno
  e benigno.
- O **K-means** encontrou exatamente os dois grupos que essa direção sugere, e a
  tabela de contingência confirma a correspondência com os diagnósticos.

Isso mostra que a rotulação clínica (maligno/benigno) não é um conceito
arbitrário sobreposto aos dados: ela está codificada na própria distribuição das
30 features. Como as variáveis mais correlacionadas descrevem o tamanho e a
irregularidade dos núcleos, os tumores malignos aparecem como um agrupamento com
valores sistematicamente mais altos nessas medidas.

### 6.6 Limitações

O K-means assume clusters esféricos de variância parecida, o que não é
exatamente verdade aqui. Além disso, o k foi fixado em 2 por conhecimento do
domínio; o método não "descobre" o número de clusters sozinho, e o agrupamento
fica sujeito à inicialização (mitigada com `n_init=10`). A visualização usa
apenas 2 dos 10+ componentes relevantes, então parte da estrutura fica fora do
gráfico. Ainda assim, a concordância entre PCA, K-means e os diagnósticos reais
é a evidência principal de que os métodos não supervisionados capturaram a
estrutura subjacente dos dados.